# fio Runtime Storage Analysis

This notebook reads a **combined benchmark output file** and filters only `storage` experiment trials before analysis.

The filtering boundary is:

```python
trial['experimentName'] == 'storage'
```

It then parses fio JSON only from the selected storage trials.


In [ ]:
from pathlib import Path
import json
import re

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', lambda value: f'{value:,.3f}')


In [ ]:
# Both notebooks can read the same combined harness output file.
RESULT_FILE = Path('run.json')
EXPERIMENT_NAME = 'storage'

with RESULT_FILE.open('r', encoding='utf-8') as file:
    harness_result = json.load(file)

all_trials = harness_result.get('trials', [])
selected_trials = [
    result for result in all_trials
    if result.get('trial', {}).get('experimentName') == EXPERIMENT_NAME
]

storage_harness = {
    **harness_result,
    'trials': selected_trials,
}

print('Run ID:', harness_result.get('runId'))
print('All trials:', len(all_trials))
print(f'{EXPERIMENT_NAME.capitalize()} trials selected:', len(selected_trials))
print('Other experiment trials ignored:', len(all_trials) - len(selected_trials))

assert selected_trials, f'No {EXPERIMENT_NAME!r} trials found in {RESULT_FILE}'
assert all(
    result.get('trial', {}).get('experimentName') == EXPERIMENT_NAME
    for result in selected_trials
)


In [ ]:
pd.DataFrame([
    {
        'trial_id': result.get('trial', {}).get('id'),
        'experiment': result.get('trial', {}).get('experimentName'),
        'runtime': result.get('trial', {}).get('runtimeName'),
        'workload': result.get('trial', {}).get('workloadName'),
        'selected': result.get('trial', {}).get('experimentName') == EXPERIMENT_NAME,
    }
    for result in all_trials
])


## Parser

The fio result is JSON encoded as a string inside:

```text
trials[*].runtimeStages[stage == "start_task"].data.stdout
```

The parser handles ANSI/control characters and rejects output that is not a valid fio result.

In [ ]:
ANSI_ESCAPE = re.compile(r'\x1B(?:[@-Z\\-_]|\[[0-?]*[ -/]*[@-~])')


def clean_terminal_output(value: str) -> str:
    """Remove common ANSI terminal escape sequences and surrounding whitespace."""
    return ANSI_ESCAPE.sub('', value or '').strip()


def get_stage(trial_result: dict, stage_name: str) -> dict | None:
    return next(
        (stage for stage in trial_result.get('runtimeStages', [])
         if stage.get('stage') == stage_name),
        None,
    )


def parse_fio_stdout(stdout: str) -> tuple[dict | None, str | None]:
    cleaned = clean_terminal_output(stdout)
    if not cleaned:
        return None, 'start_task stdout is empty'

    # Usually stdout is exactly JSON. If terminal text precedes it, try the
    # substring beginning at the first opening brace as a fallback.
    candidates = [cleaned]
    first_brace = cleaned.find('{')
    if first_brace > 0:
        candidates.append(cleaned[first_brace:])

    last_error = None
    for candidate in candidates:
        try:
            parsed = json.loads(candidate)
            if isinstance(parsed, dict) and isinstance(parsed.get('jobs'), list):
                return parsed, None
            return None, 'stdout is JSON but does not contain an fio jobs array'
        except json.JSONDecodeError as error:
            last_error = error

    return None, f'stdout is not valid fio JSON: {last_error}'


def percentile_ns(io_section: dict, percentile: str) -> float:
    return io_section.get('clat_ns', {}).get('percentile', {}).get(percentile, np.nan)


def extract_fio_rows(harness: dict) -> tuple[pd.DataFrame, pd.DataFrame]:
    valid_rows = []
    invalid_rows = []

    for result in harness.get('trials', []):
        trial = result.get('trial', {})
        runtime = trial.get('runtimeName', 'unknown')
        start_stage = get_stage(result, 'start_task')

        base = {
            'trial_id': trial.get('id'),
            'runtime': runtime,
            'runtime_handler': trial.get('runtimeHandler'),
            'workload': trial.get('workloadName'),
            'snapshotter': trial.get('Snapshotter'),
            'image': trial.get('image'),
            'harness_status': result.get('status'),
        }

        if not start_stage:
            invalid_rows.append({**base, 'reason': 'start_task stage is missing'})
            continue

        stage_data = start_stage.get('data') or {}
        fio_result, error = parse_fio_stdout(stage_data.get('stdout', ''))
        if error:
            invalid_rows.append({
                **base,
                'exit_code': stage_data.get('exit_code'),
                'reason': error,
                'stderr': stage_data.get('stderr', ''),
                'stdout_preview': clean_terminal_output(stage_data.get('stdout', ''))[:300],
            })
            continue

        for job_index, job in enumerate(fio_result.get('jobs', [])):
            if job.get('error', 0) != 0:
                invalid_rows.append({
                    **base,
                    'exit_code': stage_data.get('exit_code'),
                    'reason': f"fio job returned error={job.get('error')}",
                })
                continue

            read = job.get('read', {})
            write = job.get('write', {})
            options = job.get('job options', {})

            read_bw_mib_s = read.get('bw_bytes', 0) / (1024 ** 2)
            write_bw_mib_s = write.get('bw_bytes', 0) / (1024 ** 2)
            read_iops = read.get('iops', 0.0)
            write_iops = write.get('iops', 0.0)

            valid_rows.append({
                **base,
                'job_index': job_index,
                'job_name': job.get('jobname'),
                'fio_version': fio_result.get('fio version'),
                'rw': options.get('rw'),
                'block_size': options.get('bs'),
                'configured_size': options.get('size'),
                'runtime_seconds': job.get('job_runtime', 0) / 1000,
                'read_iops': read_iops,
                'write_iops': write_iops,
                'total_iops': read_iops + write_iops,
                'read_throughput_mib_s': read_bw_mib_s,
                'write_throughput_mib_s': write_bw_mib_s,
                'total_throughput_mib_s': read_bw_mib_s + write_bw_mib_s,
                'read_ios': read.get('total_ios', 0),
                'write_ios': write.get('total_ios', 0),
                'read_mean_latency_us': read.get('lat_ns', {}).get('mean', np.nan) / 1000,
                'write_mean_latency_us': write.get('lat_ns', {}).get('mean', np.nan) / 1000,
                'read_p50_latency_us': percentile_ns(read, '50.000000') / 1000,
                'read_p95_latency_us': percentile_ns(read, '95.000000') / 1000,
                'read_p99_latency_us': percentile_ns(read, '99.000000') / 1000,
                'write_p50_latency_us': percentile_ns(write, '50.000000') / 1000,
                'write_p95_latency_us': percentile_ns(write, '95.000000') / 1000,
                'write_p99_latency_us': percentile_ns(write, '99.000000') / 1000,
                'user_cpu_percent': job.get('usr_cpu', np.nan),
                'system_cpu_percent': job.get('sys_cpu', np.nan),
                'context_switches': job.get('ctx', np.nan),
                'start_stage_latency_ms': stage_data.get('latency_ms', np.nan),
            })

    return pd.DataFrame(valid_rows), pd.DataFrame(invalid_rows)


metrics_df, invalid_df = extract_fio_rows(storage_harness)


## Valid fio results

In [ ]:
summary_columns = [
    'runtime', 'snapshotter', 'rw', 'block_size',
    'read_iops', 'write_iops', 'total_iops',
    'read_throughput_mib_s', 'write_throughput_mib_s',
    'total_throughput_mib_s',
    'read_p99_latency_us', 'write_p99_latency_us',
    'user_cpu_percent', 'system_cpu_percent',
]

metrics_df[summary_columns].sort_values('runtime')

## Invalid or non-fio runs

These must not be included in runtime averages or graphs.

In [ ]:
if invalid_df.empty:
    print('No invalid runs detected.')
else:
    display(invalid_df)

## Throughput comparison

Read and write throughput are shown separately. The total is useful for this 50/50 mixed workload, but should not replace the directional values.

In [ ]:
throughput_plot = (
    metrics_df.set_index('runtime')
    [['read_throughput_mib_s', 'write_throughput_mib_s']]
    .rename(columns={
        'read_throughput_mib_s': 'Read',
        'write_throughput_mib_s': 'Write',
    })
)

ax = throughput_plot.plot(kind='bar', figsize=(9, 5))
ax.set_title('fio throughput by runtime')
ax.set_xlabel('Runtime')
ax.set_ylabel('Throughput (MiB/s)')
ax.tick_params(axis='x', rotation=0)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

## IOPS comparison

In [ ]:
iops_plot = (
    metrics_df.set_index('runtime')
    [['read_iops', 'write_iops']]
    .rename(columns={'read_iops': 'Read', 'write_iops': 'Write'})
)

ax = iops_plot.plot(kind='bar', figsize=(9, 5))
ax.set_title('fio IOPS by runtime')
ax.set_xlabel('Runtime')
ax.set_ylabel('IOPS')
ax.tick_params(axis='x', rotation=0)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

## Tail-latency comparison

Lower is better. p99 is usually more informative than mean latency for identifying stalls and jitter.

In [ ]:
latency_plot = (
    metrics_df.set_index('runtime')
    [['read_p99_latency_us', 'write_p99_latency_us']]
    .rename(columns={
        'read_p99_latency_us': 'Read p99',
        'write_p99_latency_us': 'Write p99',
    })
)

ax = latency_plot.plot(kind='bar', figsize=(9, 5))
ax.set_title('fio p99 latency by runtime')
ax.set_xlabel('Runtime')
ax.set_ylabel('Latency (µs)')
ax.tick_params(axis='x', rotation=0)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

## Relative performance against a baseline

This expresses each runtime as a percentage of the selected baseline. For throughput and IOPS, values above 100% are better.

In [ ]:
BASELINE_RUNTIME = 'runc'

if BASELINE_RUNTIME not in set(metrics_df['runtime']):
    raise ValueError(f'Baseline runtime {BASELINE_RUNTIME!r} is not present')

baseline = metrics_df.loc[metrics_df['runtime'] == BASELINE_RUNTIME].iloc[0]
relative_df = metrics_df[['runtime', 'total_iops', 'total_throughput_mib_s']].copy()
relative_df['iops_vs_baseline_percent'] = (
    relative_df['total_iops'] / baseline['total_iops'] * 100
)
relative_df['throughput_vs_baseline_percent'] = (
    relative_df['total_throughput_mib_s'] /
    baseline['total_throughput_mib_s'] * 100
)
relative_df.sort_values('throughput_vs_baseline_percent', ascending=False)

## Repeated-trial aggregation

When the harness contains several runs per runtime, use the mean together with standard deviation and sample count. Do not rely on a single run for final conclusions.

In [ ]:
aggregate_df = (
    metrics_df.groupby('runtime', as_index=False)
    .agg(
        samples=('trial_id', 'count'),
        mean_read_iops=('read_iops', 'mean'),
        std_read_iops=('read_iops', 'std'),
        mean_write_iops=('write_iops', 'mean'),
        std_write_iops=('write_iops', 'std'),
        mean_total_iops=('total_iops', 'mean'),
        std_total_iops=('total_iops', 'std'),
        mean_total_throughput_mib_s=('total_throughput_mib_s', 'mean'),
        std_total_throughput_mib_s=('total_throughput_mib_s', 'std'),
        mean_read_p99_latency_us=('read_p99_latency_us', 'mean'),
        mean_write_p99_latency_us=('write_p99_latency_us', 'mean'),
    )
)

aggregate_df